# 30.05 Масштабные законы двуслойной модели

Одна и та же нормированная чувствительность к сопротивлению лёгкого может соответствовать разным абсолютным импедансам. Это различие существенно при выборе размера боковой сборки: сохранение чувствительности ещё не означает сохранения измеримого сигнала. Здесь рассматривается вопрос, при каких условиях увеличение толщины верхнего слоя можно компенсировать пропорциональным увеличением сборки и какие свойства отклика при этом сохраняются.

Исходным объектом служит идеальная плоская двухслойная среда с точечными электродами. Удельные сопротивления, толщины и размеры задаются как синтетические параметры. Записи добровольцев и индивидуальная КТ в расчёте не используются. Прямая формула описана в [описании прямой модели (30.01)](30.01_Прямая_двуслойная_модель_боковой_сборки.md); её реализация и отдельные самопроверки относятся к [проверке вычислительного ядра (30.04)](30.04_Вычислительное_ядро_двуслойной_модели.ipynb).

**Основное аналитическое следствие.** При геометрическом подобии нормированная чувствительность сохраняется, а импеданс обратно пропорционален масштабу длин. Поэтому пропорциональность минимального размера толщине следует только из безразмерного критерия при неизменных отношениях сопротивлений и электродных расстояний. Для порога в омах такое правило автоматически не выполняется.

В этой редакции сохранённых численных выходов нет. Далее изложены основания трёх проверок, их параметры и критерии принятия. Выполнение кода, соблюдение допусков и численные значения результатов по этому файлу не подтверждены.


## 1. Геометрическое подобие и нормировка импеданса

Верхний слой имеет эффективное удельное сопротивление $\rho_1$ и толщину $h$, нижнее полупространство — сопротивление $\rho_2$. Оба сопротивления выражены в Ом·м. Токовые электроды расположены в точках $\pm a$, потенциальные — в точках $\pm b$ одной прямой на плоской поверхности; $0<b<a$, а внешняя база сборки равна $L=2a$. В расчёте все длины задаются в метрах. Передаточный импеданс $Z$ является вещественной знаковой величиной в омах. Реальная кривизна, конечная площадь электродов и контактные эффекты в эту постановку не входят.

Чтобы отделить влияние абсолютного размера от формы, введём безразмерные отношения $\beta=b/a$, $\eta=h/a$ и $r=\rho_2/\rho_1$. Прямая формула принимает вид

<a id="eq-dimensionless-impedance"></a>
$$
Z=\frac{\rho_1}{\pi a}\,\Psi(r,\eta,\beta), \tag{1}
$$

где $Z$ — передаточный импеданс, Ом; $\rho_1$ — сопротивление верхнего слоя, Ом·м; $a$ — половина токовой базы, м; $\Psi$ — безразмерная функция отношений сопротивлений $r$, толщины $\eta$ и электродных расстояний $\beta$. Для выбранной плоской модели эта функция записывается как

<a id="eq-dimensionless-series"></a>
$$
\Psi=\left(\frac1{1-\beta}-\frac1{1+\beta}\right)
+2\sum_{i=1}^{\infty}k^i\left[
\frac1{\sqrt{(1-\beta)^2+(2i\eta)^2}}-
\frac1{\sqrt{(1+\beta)^2+(2i\eta)^2}}
\right], \tag{2}
$$

где $k=(r-1)/(r+1)$ — безразмерный коэффициент контраста двух слоёв; $i$ — положительный целый индекс члена ряда; $r=\rho_2/\rho_1$, $\eta=h/a$ и $\beta=b/a$ — безразмерные отношения, причём сопротивления и толщина положительны, а $0<\beta<1$.

По формуле (1) одновременное увеличение $a$, $b$ и $h$ в $c$ раз при $c>0$ уменьшает $Z$ в $c$ раз: аргументы $\Psi$ не меняются. Одновременное умножение обоих сопротивлений на положительный множитель $q$ умножает $Z$ на $q$. Последнее утверждение является следствием формулы; отдельного численного опыта с множителем $q$ в приведённом коде нет.

Для сопоставления чувствительности используется безразмерная эластичность $S_{\rho_2}=(\rho_2/Z)\,\partial Z/\partial\rho_2$. Она характеризует относительное изменение импеданса при малом относительном изменении сопротивления нижнего слоя и фиксированных остальных параметрах. По формулам (1)–(2) эта величина определяется отношениями $r$, $\eta$ и $\beta$.

Численная проверка ниже задаёт $\rho_1=6$ Ом·м, $\rho_2=18$ Ом·м и $\beta=0{,}5$. Для каждого $\eta$ из набора $0{,}3$; $0{,}6$; $1{,}0$; $2{,}0$ сравниваются $a=0{,}050$ и $0{,}150$ м; при этом $h=\eta a$ и $b=\beta a$. Проверяются совпадение нормированного импеданса $Za/\rho_1$ и совпадение $S_{\rho_2}$ с относительным допуском $10^{-10}$ и абсолютным допуском $10^{-12}$. Оба допуска относятся к безразмерным величинам и служат проверке реализации формулы.

Безразмерная форма отделяет геометрическое подобие от изменения относительной глубины. Следующая задача состоит в проверке того, как убывает чувствительность к нижнему слою при увеличении $\eta$; сравнение двух подобных геометрий само по себе этого убывания не устанавливает.


In [1]:
from pathlib import Path
import sys

import numpy as np

candidates = [Path.cwd(), Path.cwd() / "Colab Notebooks", Path.cwd().parent]
notebook_root = next((p for p in candidates if (p / "two_layer_model.py").exists()), None)
if notebook_root is None:
    raise FileNotFoundError("two_layer_model.py not found; run from the repository or Colab Notebooks directory")
sys.path.insert(0, str(notebook_root))

from two_layer_model import evaluate, geometry_from_size, transfer_impedance

In [2]:
rho1, rho2, beta = 6.0, 18.0, 0.5
for eta in (0.3, 0.6, 1.0, 2.0):
    values = []
    sensitivities = []
    for a in (0.050, 0.150):
        h, b = eta * a, beta * a
        result = evaluate(rho1, rho2, h, a, b)
        values.append(result.z * a / rho1)
        sensitivities.append(rho2 / result.z * result.d_rho2)
    assert np.allclose(values[0], values[1], rtol=1e-10, atol=1e-12)
    assert np.allclose(sensitivities[0], sensitivities[1], rtol=1e-10, atol=1e-12)
    print(eta, values[0], sensitivities[0])

0.3 0.6862074832878098 0.35973121119821994
0.6 0.5141961216733381 0.1502461084698273
1.0 0.4541719605396324 0.0545545486587862
2.0 0.4292260817299911 0.00917434658948949


## 2. Убывание чувствительности при большой относительной глубине

Для проверки подобия предусмотрены вывод нормированного импеданса и эластичности, а также сравнения с заданными допусками. Эти выходы не сохранены, поэтому численное совпадение здесь не констатируется. Следующий опыт использует аналитическую формулу и исследует изменение относительной толщины при фиксированной токовой базе.

При $\eta\gg1$ ведущий член разности двух размерных ядер каждого изображения равен $ab/(4i^3h^3)$, где $a$, $b$ и $h$ выражены в метрах, а $i$ обозначает номер изображения. Эта разность имеет размерность обратной длины. При фиксированных $r$ и $\beta$ соответствующая поправка к безразмерной функции $\Psi$ убывает как $\eta^{-3}$. Для эластичности $S_{\rho_2}$ исходная асимптотика также задаёт кубическое убывание. Это предельное утверждение идеальной модели; оно не задаёт численно границу, начиная с которой асимптотика достаточно точна.

В коде сохраняются $\rho_1=6$ Ом·м, $\rho_2=18$ Ом·м и $\beta=0{,}5$, а $a$ фиксируется равным $0{,}050$ м. Значения $\eta$ составляют $2$, $4$, $8$, $16$ и $32$; толщина в каждой точке равна $h=\eta a$. По соседним точкам вычисляется отношение приращения логарифма $S_{\rho_2}$ к приращению логарифма $\eta$. Для кубического закона это отношение должно стремиться к $-3$.

Критерий кода требует, чтобы последний такой наклон отличался от $-3$ менее чем на $0{,}01$. Проверяется только последняя пара точек; прохождение этого условия не удостоверяло бы точность асимптотики на всей сетке. Значения чувствительности и наклонов должны выводиться отдельно. Их сохранённых значений в файле нет.

Асимптотика связывает ослабление чувствительности с относительной толщиной. Чтобы использовать эту зависимость для выбора размера, необходимо задать критерий достаточной чувствительности. Следующий этап рассматривает безразмерный порог и отдельно проверяет, сохраняется ли при подобии абсолютный дыхательный сигнал.


In [3]:
a = 0.050
etas = np.array([2.0, 4.0, 8.0, 16.0, 32.0])
s_rho2 = []
for eta in etas:
    result = evaluate(rho1, rho2, eta * a, a, beta * a)
    s_rho2.append(rho2 / result.z * result.d_rho2)
s_rho2 = np.asarray(s_rho2)
local_slopes = np.diff(np.log(s_rho2)) / np.diff(np.log(etas))
assert abs(local_slopes[-1] + 3.0) < 0.01
print("eta:", etas)
print("S_rho2:", s_rho2)
print("local log-log slopes:", local_slopes)

eta: [ 2.  4.  8. 16. 32.]
S_rho2: [9.17434659e-03 1.24505138e-03 1.58852563e-04 1.99566798e-05
 2.49768145e-06]
local log-log slopes: [-2.88140013 -2.97044501 -2.99274474 -2.99821032]


## 3. Минимальный размер при безразмерном и абсолютном порогах

Результат проверки последнего логарифмического наклона не сохранён. Поэтому при переходе к размеру сборки используем только аналитическую зависимость от безразмерных отношений, а численное приближение к асимптотике оставляем неподтверждённым.

Если достаточность размера определяется условием $S_{\rho_2}\ge S_*$ при фиксированных $r$ и $\beta$, порог задаёт одно и то же условие на $\eta=2h/L$. Здесь $S_*$ — заранее выбранный безразмерный порог чувствительности. При наличии проходящего размера и отсутствии дополнительных ограничений отношение $L_{min}/h$ остаётся постоянным. Это условное правило выбора внутри идеальной модели.

В первом расчёте задан учебный порог $S_*=0{,}25$. Сетка отношения $\lambda=L/h$ содержит 4000 равномерно расположенных точек от $0{,}5$ до $40{,}0$. Для толщин $h=0{,}010$ и $0{,}030$ м код находит первую точку сетки, в которой эластичность достигает порога. Следовательно, найденный минимум относится к этой сетке; точный непрерывный минимум и первый допустимый размер изготовленного ряда здесь не определяются. Если порог не достигнут, выполнение должно завершиться сообщением об отсутствии решения в исследованном диапазоне.

Отношения найденных размеров к соответствующим толщинам сравниваются с относительным допуском $10^{-12}$. Сопротивления $6$ и $18$ Ом·м и отношение $\beta=0{,}5$ сохраняются из первого опыта. Порог $0{,}25$ выбран для демонстрации масштабного закона и не обоснован как требование к реальному измерению.

Абсолютный критерий $|\Delta Z|\ge c\sigma_Z$ проверяет другое свойство: различимо ли изменение импеданса на фоне заданного уровня неопределённости. Здесь $\Delta Z$ и $\sigma_Z$ выражены в омах, а $c$ — безразмерный множитель порога. В этом критерии $c$ обозначает множитель неопределённости; он не связан с масштабом длин из раздела 1. Значения $c$ и $\sigma_Z$ в расчёте не задаются. Их выбор требует отдельного обоснования для конкретного измерения; распределение погрешности здесь не предполагается.

Во втором расчёте сравниваются синтетические состояния с $\rho_2=15$ Ом·м после выдоха и $\rho_2=25$ Ом·м после вдоха при неизменном $\rho_1=6$ Ом·м. Эти названия обозначают состояния модели, а не зарегистрированные дыхательные события. Исходные $h=0{,}020$ м и $L=0{,}140$ м затем одновременно увеличиваются в три раза при сохранении $\beta$. Дыхательный сигнал определяется как разность полного импеданса после вдоха и после выдоха при одной геометрии.

По формуле (1) увеличение всех длин в три раза должно уменьшить эту разность втрое. Код сравнивает разность для увеличенной геометрии с одной третью исходной при относительном допуске $10^{-10}$. В двух сравнениях этого раздела код задаёт относительный допуск, а абсолютный допуск сравнения явно не указан и остаётся программным значением по умолчанию. Этот допуск нельзя считать установленной точностью измерения. Сохранённых результатов обоих сравнений нет.

Таким образом, безразмерный порог позволяет переносить отношение размера к толщине, но фиксированный порог в омах требует отдельной проверки. Даже успешное численное подтверждение подобия не определило бы, достаточно ли абсолютного сигнала для восстановления параметров тканей.


In [4]:
s_threshold = 0.25  # только синтетическая демонстрация масштабного тождества
lambda_grid = np.linspace(0.5, 40.0, 4000)  # lambda = L/h

def lmin_by_dimensionless_threshold(h):
    for ratio in lambda_grid:
        size = ratio * h
        a, b = geometry_from_size(size, beta)
        result = evaluate(rho1, rho2, h, a, b)
        sensitivity = rho2 / result.z * result.d_rho2
        if sensitivity >= s_threshold:
            return size
    raise RuntimeError("threshold is outside the scanned dimensionless range")

lmin_1 = lmin_by_dimensionless_threshold(0.010)
lmin_2 = lmin_by_dimensionless_threshold(0.030)
assert np.isclose(lmin_1 / 0.010, lmin_2 / 0.030, rtol=1e-12)

rho2_exhale, rho2_inhale = 15.0, 25.0
h1, size1, scale = 0.020, 0.140, 3.0
a1, b1 = geometry_from_size(size1, beta)
a2, b2 = geometry_from_size(scale * size1, beta)
delta_1 = transfer_impedance(rho1, rho2_inhale, h1, a1, b1) - transfer_impedance(rho1, rho2_exhale, h1, a1, b1)
delta_2 = transfer_impedance(rho1, rho2_inhale, scale * h1, a2, b2) - transfer_impedance(rho1, rho2_exhale, scale * h1, a2, b2)
assert np.isclose(delta_2, delta_1 / scale, rtol=1e-10)
print("Lmin/h:", lmin_1 / 0.010)
print("absolute signal ratio after x3 geometry:", delta_2 / delta_1)

Lmin/h: 4.747311827956989
absolute signal ratio after x3 geometry: 0.3333333333333333


## 4. Что масштабные законы дают для выбора сборок

Аналитическая постановка сводит влияние геометрии и сопротивлений к отношениям $r$, $\eta$ и $\beta$. Из неё следуют однородность по сопротивлениям и обратная однородность по длинам. Асимптотика большой относительной толщины описывает кубическое убывание чувствительности к нижнему слою. Пропорциональность $L_{min}$ толщине относится только к безразмерному критерию при фиксированных $r$ и $\beta$.

Эти утверждения описывают идеальную формулу. Три блока кода предназначены для проверки геометрического подобия, последнего наклона глубинной зависимости и различия безразмерного и абсолютного критериев. Поскольку выходов нет, нельзя сообщить, что проверки пройдены, назвать рассчитанные минимальные размеры или оценить достигнутую численную погрешность.

Для следующего этапа — [проверки критериев минимального размера (32.01)](32.01_Критерии_Lmin_боковых_сборок.ipynb) — существенны два условия: порог должен соответствовать выбранной величине, а результат поиска на безразмерной сетке следует отличать от выбора среди реально доступных размеров. Абсолютный порог требует обоснованной неопределённости измеренного изменения импеданса. Допустимость размера по индивидуальной геометрии проверяется отдельно от чувствительности идеальной модели.

Точность совместного восстановления сопротивлений, распределения ошибок и достаточность пары размеров из масштабных тождеств не следуют. Эти вопросы рассматриваются в серии 31, при выборе размеров в серии 32 и при анализе наблюдений в серии 33. Данный расчёт передаёт им аналитические ограничения, но не субъектные оценки и не проверенный численный артефакт.

Исторический смешанный предшественник сохранён в [архивной версии аналитического обоснования](archive/legacy/30.91_Историческое_аналитическое_обоснование.ipynb). Он служит для прослеживания происхождения текста и не заменяет отсутствующие результаты текущих проверок.
